# 📘 Agentic Architectures 2 (Agno): Tool Use

This notebook is the **Agno-framework** counterpart of `02_tool_use.ipynb`. The original implementation uses LangGraph's `StateGraph` to wire an agent node, a `ToolNode`, and a conditional edge that loops while the LLM requests tool calls. Here we re-implement exactly the same scenario — give an LLM a Tavily web search tool and let it decide when to call it — but using [Agno](https://docs.agno.com)'s high-level `Agent` abstraction, which internalises the think → act → observe loop instead of asking the user to wire it explicitly.

The LLM is reached via SiliconFlow's OpenAI-compatible endpoint (same provider used in `03_ReAct.ipynb`).

### Why a parallel implementation?

LangGraph and Agno operate at different abstraction levels:

| Concern | LangGraph (`02_tool_use.ipynb`) | Agno (this notebook) |
|---|---|---|
| Control flow | User-defined `StateGraph` with `agent` node, `ToolNode`, and a conditional edge looping back to `agent` | Internal to `Agent.run()` — the loop is hidden behind one call |
| State | `TypedDict` + `add_messages` reducer | Managed by the framework; observable via `RunOutput.messages` |
| Tool definition | LangChain `TavilySearchResults` instance, bound via `llm.bind_tools(...)` | `TavilyTools()` toolkit instance passed via `tools=[...]` |
| Driver | `app.stream(initial_input, stream_mode='values')` | `agent.run(query)` (non-streaming, deterministic timing) |

Everything else — the architecture (Tool Use), the workflow (think → act → observe), the user-facing behaviour — is identical. This makes it a clean side-by-side reference for the *framework overhead*, not the *agent design*.

## Phase 0: Foundation & Setup

### Step 0.1: Installing Core Libraries

We need `agno` itself, `openai` (its OpenAI-compatible client), `python-dotenv`, and `rich` for pretty printing. Tavily is reached through Agno's built-in `TavilyTools` toolkit — no separate `tavily-python` install is necessary because `agno.tools.tavily` already wraps the HTTP API.

In [ ]:
# !pip install -q -U agno openai python-dotenv rich pydantic

### Step 0.2: Importing Libraries and Setting Up Keys

**Action Required:** Create a `.env` file in this directory with your keys:
```
SILICONFLOW_API_KEY="sk-..."           # https://siliconflow.cn
TAVILY_API_KEY="your_tavily_api_key_here"
```

Note: We explicitly disable Agno's built-in telemetry (`telemetry=False`) for the same reason the LangGraph notebooks disable LangSmith when run as benchmarks — we don't want background HTTP calls polluting the request path.

In [ ]:
import os
from typing import Any
from dotenv import load_dotenv

# Pydantic for structured-output schemas
from pydantic import BaseModel, Field

# Agno components
from agno.agent import Agent
from agno.models.openai import OpenAILike
from agno.tools.tavily import TavilyTools

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

load_dotenv()

for key in ["SILICONFLOW_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded.")

## Phase 1: Defining the Agent's Toolkit

In LangGraph we instantiated a `TavilySearchResults` object and edited its `.name`/`.description` so the LLM would understand when to call it. Agno's `TavilyTools` toolkit already exposes a `web_search` function with a sensible description, so we just instantiate it with our API key.

In [ ]:
console = Console()

# Agno's built-in Tavily toolkit. `search_depth='advanced'` is the default and matches
# the original notebook's intent of getting substantive snippets back, not just titles.
search_tool = TavilyTools(
    api_key=os.environ.get("TAVILY_API_KEY"),
    search_depth="advanced",
    format="markdown",
    include_answer=True,
)

console.print(f"[bold green]Toolkit registered:[/bold green] {type(search_tool).__name__}")
console.print(f"[dim]Exposed functions: {[m.__name__ for m in search_tool._tool_methods() if hasattr(search_tool, '_tool_methods')]}[/dim]")

**Discussion:** In LangGraph we attached the tool to the LLM via `.bind_tools([search_tool])`. In Agno we don't do that step at all — we pass `tools=[search_tool]` straight to the `Agent` constructor and the framework handles tool-schema generation, model-side binding, the call dispatch, and the observation feedback loop. That's the central difference between the two abstractions.

## Phase 2: Building the Tool-Using Agent with Agno

### Step 2.1: Configuring the LLM

We talk to SiliconFlow through the OpenAI-compatible protocol. Agno's `OpenAILike` is exactly the right adapter — same shape as the OpenAI provider, but accepts a custom `base_url`.

Swap `id` for any tool-calling-capable model on SiliconFlow, e.g. `Qwen/Qwen2.5-72B-Instruct`, `deepseek-ai/DeepSeek-V3`, etc.

In [ ]:
model = OpenAILike(
    id="deepseek-ai/DeepSeek-V3",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
    temperature=0,
)

print(f"LLM configured: {model.id} via {model.base_url}")

### Step 2.2: Constructing the Agent

This single `Agent(...)` call is the Agno equivalent of the entire LangGraph state-machine construction: define state → create `agent_node` → create `ToolNode` → add entry point → add conditional edge → add `tools → agent` loop → compile. All of that is collapsed into the constructor.

In [ ]:
agent = Agent(
    name="Tool-Use Agent",
    model=model,
    tools=[search_tool],
    instructions=[
        "You are a helpful assistant with access to a web search tool.",
        "When the user asks about events, news, or anything beyond your training data, call the web search tool.",
        "After receiving search results, synthesise a concise, factual answer grounded in those results.",
    ],
    add_datetime_to_context=True,
    markdown=True,
    telemetry=False,  # disable Agno's own telemetry pings
    tool_call_limit=4,  # safety cap, same spirit as LangGraph's implicit termination
)

print("Tool-using Agno agent built.")
print(f"  tools: {[type(t).__name__ for t in agent.tools]}")

**Discussion of the difference:**

There is no `agent_node`, no `tool_node`, no `router_function`, no `add_conditional_edges`, no compiled `StateGraph`. Agno collapses the entire think → act → observe state machine behind `agent.run(...)`. The trade-off is symmetrical:

- **LangGraph:** explicit control flow you can see, edit, and route differently per node. Useful when the agent's loop is non-standard.
- **Agno:** zero plumbing for the common case. The price is that customising the loop (e.g. inserting a custom step *between* tool result and the next LLM turn) means dropping down to lower-level hooks or pre/post-run callbacks.

For the **Tool Use** architecture, where the loop is the textbook think-act-observe shape, Agno is the more direct expression.

## Phase 3: End-to-End Execution

### Step 3.1: Running the Agent

We run the same query the original notebook used — something the model cannot answer from training data, so it is forced to call the web search tool. `agent.run(...)` returns a `RunOutput` whose `messages` list is the full conversation trace (system / user / assistant / tool messages), letting us inspect exactly the same think → act → observe sequence we saw in the LangGraph version.

In [ ]:
user_query = "What were the main announcements from Apple's latest WWDC event?"

console.print(f"[bold cyan]🚀 Kicking off Tool Use workflow for request:[/bold cyan] '{user_query}'\n")

response = agent.run(user_query)

console.print("[bold green]✅ Tool Use workflow complete![/bold green]")
console.print("\n[bold]Final Answer:[/bold]")
console.print(Markdown(response.content or "(no content)"))

### Step 3.2: Inspecting the Execution Trace

The Agno `RunOutput` is a rich object. The two fields we care about for understanding the run are:

- `response.messages` — the full message list, equivalent to what we manually accumulated via `add_messages` in LangGraph
- `response.metrics` — token counts and timing info that LangGraph does not surface natively

The pattern below mirrors the `_extract_execution_trace` helper used in production Agno deployments.

In [ ]:
def summarise_messages(messages: list[Any]) -> None:
    """Print a compact trace of roles + tool calls, mirroring LangGraph's pretty_print loop."""
    for i, m in enumerate(messages or []):
        role = getattr(m, "role", "?")
        tool_calls = getattr(m, "tool_calls", None) or []
        content = getattr(m, "content", None)
        snippet = (content[:200] + "…") if isinstance(content, str) and len(content) > 200 else (content or "")
        console.print(f"[bold]{i:02d}[/bold] [yellow]{role}[/yellow] tool_calls={len(tool_calls)}")
        if tool_calls:
            for tc in tool_calls:
                fn = tc.get("function", {}) if isinstance(tc, dict) else {}
                console.print(f"     [cyan]→ {fn.get('name')} args={fn.get('arguments')}[/cyan]")
        if snippet:
            console.print(f"     {snippet}")

console.print("[bold]--- Execution Trace ---[/bold]")
summarise_messages(response.messages)

if getattr(response, "metrics", None):
    m = response.metrics
    console.print("\n[bold]--- Metrics ---[/bold]")
    console.print({
        "input_tokens": getattr(m, "input_tokens", None),
        "output_tokens": getattr(m, "output_tokens", None),
        "total_tokens": getattr(m, "total_tokens", None),
        "response_time": getattr(m, "response_time", None),
    })

**Discussion:** The trace tells the same story as the LangGraph version:

1. A `system` message carrying the instructions and (because we set `add_datetime_to_context=True`) the current timestamp
2. A `user` message with the query
3. An `assistant` message with `tool_calls` populated — the LLM decided to search
4. A `tool` message with the Tavily search results
5. A final `assistant` message synthesising the answer

This is the same think → act → observe sequence implemented in LangGraph as `agent → call_tool → agent`. The shape is identical; what changes is who manages the bookkeeping.

## Phase 4: Evaluation with LLM-as-a-Judge

We preserve the original notebook's LLM-as-a-Judge cell, but use Agno's native structured output (`output_schema=...`, `use_json_mode=True`) instead of LangChain's `.with_structured_output(...)`. This is exactly the pattern the production `agent_service` router uses.

In [ ]:
class ToolUseEvaluation(BaseModel):
    """Schema for evaluating the agent's tool use and final answer."""
    tool_selection_score: int = Field(description="Score 1-5 on whether the agent chose the correct tool for the task.")
    tool_input_score: int = Field(description="Score 1-5 on how well-formed and relevant the input to the tool was.")
    synthesis_quality_score: int = Field(description="Score 1-5 on how well the agent integrated the tool's output into its final answer.")
    justification: str = Field(description="A brief justification for the scores.")

judge = Agent(
    name="Judge",
    model=model,  # reuse the same LLM client
    instructions=[
        "You are an expert judge of AI agents.",
        "Score the trace on a scale of 1-5 for each criterion and provide a brief justification.",
    ],
    output_schema=ToolUseEvaluation,
    use_json_mode=True,
    telemetry=False,
)

# Reconstruct a textual trace from the messages so the judge can read it
trace_text = "\n".join(
    f"{getattr(m, 'role', '?')}: {getattr(m, 'content', '') or ''} tool_calls={getattr(m, 'tool_calls', None) or ''}"
    for m in (response.messages or [])
)

judge_response = judge.run(
    f"Evaluate the following conversation trace based on the agent's tool use.\n\nTrace:\n```\n{trace_text}\n```"
)

console.print("[bold]--- Evaluation ---[/bold]")
console.print(judge_response.content)

## Conclusion

We re-implemented the **Tool Use** architecture in Agno. The agent has the same observable behaviour as the LangGraph version: it receives a query, calls Tavily, observes the results, and synthesises a final answer. What changed is *who writes the loop*:

- **LangGraph:** the developer explicitly wires `agent → router → tool → agent` as a `StateGraph` and compiles it.
- **Agno:** the developer passes `tools=[...]` to a single `Agent` constructor; the loop is internal to `agent.run()`.

For benchmark purposes the two implementations are directly comparable: the same Tavily tool, the same LLM, the same query — only the orchestrator changes. The pattern continues in `03_ReAct_agno.ipynb` and `11_meta_controller_agno.ipynb`.